In [ ]:
import os
import sys
import re

import numpy as np
import matplotlib.pyplot as plt
import tidy3d as td
from dotenv import load_dotenv

load_dotenv()

# Add parent project path so AutomationModule is importable
sys.path.append(os.path.abspath(fr'H:\phd stuff\tidy3d'))
from AutomationModule import *
import AutomationModule as AM

tidy3dAPI = os.environ["API_TIDY3D_KEY"]

In [ ]:

field_folder = r"./data/20260422_far_field_n_3.4_ff_0.2172_ffh_0.225_schulz.h5"
ref_field_folder = r"./data/20260422_far_field_n_3.4_ff_0.2172_ffh_0.225_schulz_reference.h5"
param_folder = r"./data/20260422_far_field_n_3.4_ff_0.2172_ffh_0.225_schulz_params.h5"

In [ ]:
field_ref = AM.read_hdf5_as_dict(ref_field_folder)
field = AM.read_hdf5_as_dict(field_folder)
param = AM.read_hdf5_as_dict(param_folder)

In [ ]:
def solid_angle_weights(theta):
    """Compute per-bin solid-angle weights: w_i = sin(theta_i) * Delta_theta_i.
    The phi factor (Delta_phi) is multiplied externally."""
    return np.sin(theta) * np.gradient(theta)


def compute_transmission(
    E_theta_ref, E_phi_ref,    # shape (n_theta, n_phi, n_freq), complex — reference (air) far field
    E_theta_scat, E_phi_scat,  # same shape — scattered (slab) far field
    theta, phi           # 1D arrays in radians
):
    """ Returns:
        T_ballistic: coherent overlap with the reference angular mode
        T_co: total co-polarized power in the angular grid
        T_cross: total cross-polarized power in the angular grid
    """
   

    # --- Build 2-D solid-angle weight grid: W(theta, phi) = sin(theta)*dtheta*dphi ---
    dphi = np.diff(phi)[1]                              # uniform phi spacing
    w_theta = solid_angle_weights(theta)    # shape (n_theta_masked,)
    W = np.outer(w_theta, np.ones_like(phi)) * dphi     # shape (n_theta_masked, n_phi)

    nfreq = E_theta_ref.shape[-1] if E_theta_ref is not None else E_phi_ref.shape[-1]
    T_ballistic = np.zeros(nfreq, dtype=float)
    T_co = np.zeros(nfreq, dtype=float)
    T_cross = np.zeros(nfreq, dtype=float)

    for k in range(nfreq):
        # Stack (E_theta, E_phi) into a 2-component vector at each angle — shape (nt, np, 2)
        Eref  = np.stack([E_theta_ref[:, :, k],  E_phi_ref[:, :, k]],  axis=-1)
        Escat = np.stack([E_theta_scat[:, :, k], E_phi_scat[:, :, k]], axis=-1)

        # Reference power per angular bin: P_ref = |E_ref|^2
        P_ref = np.sum(Eref * np.conj(Eref), axis=-1).real

        # Inner product E_scat · E_ref* (complex, used for projection)
        dot_scat_ref = np.sum(Escat * np.conj(Eref), axis=-1)

        # Avoid division by zero where the reference field is negligible
        mask = P_ref > 0
        sqrtPref = np.zeros_like(P_ref)
        sqrtPref[mask] = np.sqrt(P_ref[mask])

        # Co-polarized field: E_co = (E_scat · E_ref*) / |E_ref|
        E_co = np.zeros_like(dot_scat_ref)
        E_co[mask] = dot_scat_ref[mask] / sqrtPref[mask]
        P_co = np.abs(E_co)**2

        # Total scattered power
        P_scat = np.sum(Escat * np.conj(Escat), axis=-1).real

        # Cross-polarized power: P_cross = |E_scat|^2 - P_co
        P_cross = np.maximum(P_scat - P_co, 0.0)

        # --- Solid-angle integration ---
        T_co[k]    = np.sum(P_co    * W) / np.sum(P_ref * W)
        T_cross[k] = np.sum(P_cross * W) / np.sum(P_ref * W)

        # Ballistic: A = integral(E_scat·E_ref* dΩ) / integral(|E_ref|^2 dΩ), T_bal = |A|^2
        A = np.sum(dot_scat_ref * W) / np.sum(P_ref * W)
        T_ballistic[k] = np.abs(A)**2

    return T_ballistic, T_co, T_cross

In [ ]:
field.keys()

In [ ]:
components_data = {}

for n in field.keys():
    for f in field[n].keys():
        for z in field[n][f].keys():
            for size in field[n][f][z].keys():
               for sample in field[n][f][z][size].keys():
                    T_ballistic, T_co, T_cross = compute_transmission(
                        field_ref["Etheta_reference"].squeeze(),
                        field_ref["Ephi_reference"].squeeze(),
                        field[n][f][z][size][sample]["Etheta"].squeeze(),
                        field[n][f][z][size][sample]["Ephi"].squeeze(),
                        param["theta_proj"],
                        param["phi_proj"],
                    )
                    if n not in components_data.keys():
                        components_data[n] = {}
                    if f not in components_data[n].keys():
                        components_data[n][f] = {}
                    if z not in components_data[n][f].keys():
                        components_data[n][f][z] = {}
                    if size not in components_data[n][f][z].keys():
                        components_data[n][f][z][size] = {}
                    if sample not in components_data[n][f][z][size].keys():
                        components_data[n][f][z][size][sample] = {}

                    components_data[n][f][z][size][sample] = {
                        "T_ballistic": T_ballistic,
                        "T_co": T_co,
                        "T_cross": T_cross,
                    }

In [ ]:
titles = [r"$T_{\mathrm{ballistic}}$", r"$T_{\mathrm{co}}$", r"$T_{\mathrm{cross}}$"]
component_keys = ["T_ballistic", "T_co", "T_cross"]
lambdas = param["lambdas"]
for n in components_data.keys():
    for f in components_data[n].keys():
        for z in components_data[n][f].keys():
            for size in components_data[n][f][z].keys():
               for sample in components_data[n][f][z][size].keys():
                   plt.plot(lambdas, components_data[n][f][z][size][sample]["T_ballistic"], label=titles[0])
                   plt.plot(lambdas, components_data[n][f][z][size][sample]["T_co"], label=titles[1])
                   plt.plot(lambdas, components_data[n][f][z][size][sample]["T_cross"], label=titles[2])
                   plt.title(f"n={n}, f={f}, z={z}, size={float(size)*11.44:.2f}, sample={sample}")
                   plt.legend()
                   plt.yscale("log")
                   plt.show()